This notebook demonstrates how to run the forest deforestation User Defined Process

In [ ]:
import logging

from utils import urls

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
spatial_extent = {
    "west": 30.5503711040000994,
    "south": 1.0709279050000799,
    "east": 31.2229521229999989,
    "north": 1.5469373050000299,
}

resample_spatial_resolution = 30  # m

temporal_variability_threshold = 0.5  # units: dB
flattening_threshold = 0.12  # units: dimensionless
logistic_sse_threshold = 18.3  # units: dB^2

min_connected_area = 10000  # m^2

In [ ]:
# more restrictive (excludes more pixels from being detected as deforestation) than default
temporal_variability_threshold = 0.6
flattening_threshold = 0.14

## Sentinel 1

In [ ]:
# load results from previous batch job
JOB_ID = "j-2607241322294e99a2aaac75a3fc3708"

s1_dB = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
# load_stac adds a time dimension 😠
s1_dB = s1_dB.drop_dimension("t")

# load forest baseline

In [ ]:
# load results from previous batch job
JOB_ID = "j-26072410014341f0b1905a575d801af1"

forest_baseline_mask = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
# load_stac adds a time dimension 😠
forest_baseline_mask = forest_baseline_mask.drop_dimension("t")

In [ ]:
# load_stac incorrectly sets nodata=0 😠
forest_baseline_mask = forest_baseline_mask == 1

# Run deforestation UDP

In [ ]:
cube = connection.datacube_from_process(
    "deforestation",
    namespace=urls.DEFORESTATION_UDP,
    # spatial_extent=spatial_extent,
    forest_baseline_datacube=forest_baseline_mask,
    sentinel_1_datacube=s1_dB,
    temporal_variability_threshold=temporal_variability_threshold,
    flattening_threshold=flattening_threshold,
)

In [ ]:
job = cube.create_job(out_format="GTiff")
# job = cube.create_job(out_format="netCDF")

In [ ]:
job.start_and_wait()

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-udp/
!rm -r output-udp/

In [ ]:
results.download_files("output-udp/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)